# Parameter-Efficient Fine-Tuning with QLoRA: Educational Model Specialization
**Module 7 Project**

---

**Name:** Guillermo Martín Limón Molina

**Institution:** Universidad Politécnica de Baja California

**Date:** 11/18/2026

---

## Project Overview

This project focuses on implementing Parameter-Efficient Fine-Tuning (PEFT) using QLoRA to create a specialized educational model. You will:

1. **Select and justify** a base model and educational domain for specialization
2. **Curate a high-quality dataset** specific to your chosen educational domain
3. **Implement QLoRA fine-tuning** using Hugging Face PEFT library
4. **Evaluate and compare** your fine-tuned model against the base model
5. **Document the process** and prepare the model for educational deployment



## Step 1: Domain selection and project planning

**Choose your educational specialization domain:**

Options to consider:
- **Mathematics tutoring** - Algebra, calculus, problem-solving explanations
- **Programming education** - Code explanation, debugging help, concept teaching
- **Scientific writing** - Research paper assistance, academic language
- **Language learning** - Grammar correction, vocabulary building, conversation practice
- **History education** - Historical analysis, source interpretation, timeline explanations
- **Creative writing** - Story development, writing feedback, style guidance

**Your chosen domain:** Scientific writing.

**Why this domain needs specialization:** My scientific work is very specific, so the task should be.

**Target audience (grade level/expertise):** Researches that write scientific articles.

**Specific capabilities you want to develop:** Create my personal model who can assist me in writing scientific papers.


## Step 2: Environment setup and library installation

Install and import the required libraries for QLoRA fine-tuning.


In [ ]:
!pip install transformers peft trl datasets accelerate bitsandbytes wandb

In [ ]:
!pip install --upgrade pip setuptools wheel

In [ ]:
!pip install "cython<3.0.0"

In [3]:
%pip install "pyyaml>=6.0.2"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Install required libraries (run this if not already installed)
# !pip install transformers peft trl datasets accelerate bitsandbytes wandb

# Import essential libraries
import torch
import pandas as pd
import json
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import warnings
warnings.filterwarnings('ignore')

# Check if CUDA is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


ModuleNotFoundError: No module named 'torch'

## Step 3: Base model selection and configuration

Choose and configure your base model for fine-tuning.


In [ ]:
# Choose your base model (uncomment one)
# Option 1: Mistral-7B (Recommended for most tasks)
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

# Option 2: Llama-3-8B (Good for conversational tasks)
# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

# Option 3: Phi-3-mini (Lighter option for limited resources)
# model_name = "microsoft/Phi-3-mini-4k-instruct"

print(f"Selected base model: {model_name}")

# Configure quantization for QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Tokenizer loaded successfully!")


## Step 4: Dataset curation and preparation

Create or load your domain-specific dataset for fine-tuning.


In [ ]:
# Load a suitable educational dataset for your domain
# Choose ONE of these options:

# Option 1: General instruction-following dataset
# from datasets import load_dataset
# dataset = load_dataset("databricks/databricks-dolly-15k")

# Option 2: Mathematics-focused dataset
# dataset = load_dataset("microsoft/orca-math-word-problems-200k")

# Option 3: Code/Programming dataset
# dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca")

# TODO: Load your chosen dataset here
# dataset = load_dataset("your-chosen-dataset")

# TODO: Filter/subset the dataset for your specific domain (aim for 500-1000 examples)
# Example: dataset = dataset.filter(lambda x: "math" in x["instruction"].lower())

print("Dataset loading instructions:")
print("1. Uncomment one of the dataset options above")
print("2. Run the cell to load the dataset")
print("3. Filter the dataset to your educational domain")
print("4. Aim for 500-1000 high-quality examples")


In [ ]:
# TODO: Format your dataset for training
# The dataset should have 'instruction' and 'response' or 'input' and 'output' columns
# Example formatting function:

def format_instruction(sample):
    """Format training examples - modify based on your dataset structure"""
    # TODO: Adjust this based on your dataset's column names
    instruction = sample.get('instruction', sample.get('input', ''))
    response = sample.get('response', sample.get('output', ''))
    return f"<s>[INST] {instruction} [/INST] {response}</s>"

# TODO: Apply formatting and create train/test split
# formatted_dataset = dataset.map(lambda x: {"text": format_instruction(x)})
# split_dataset = formatted_dataset.train_test_split(test_size=0.1, seed=42)

print("Dataset formatting instructions:")
print("1. Modify the format_instruction function based on your dataset structure")
print("2. Apply the formatting to your loaded dataset")
print("3. Create train/validation split")
print("4. Verify the formatting looks correct")


## Step 5: Load model and configure QLoRA

Load the base model with quantization and set up LoRA configuration.


In [ ]:
# TODO: Load the base model with quantization
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True,
# )

# TODO: Prepare model for k-bit training
# model = prepare_model_for_kbit_training(model)

# TODO: Configure LoRA parameters
# Experiment with these values:
# - r (rank): 8, 16, 32 (higher = more parameters)
# - lora_alpha: 16, 32, 64 (scaling parameter)
# - lora_dropout: 0.05, 0.1, 0.2 (regularization)

# lora_config = LoraConfig(
#     r=16,
#     lora_alpha=32,
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
#     lora_dropout=0.1,
#     bias="none",
#     task_type="CAUSAL_LM"
# )

# TODO: Apply LoRA to the model
# model = get_peft_model(model, lora_config)

print("Model loading instructions:")
print("1. Load your chosen base model with quantization")
print("2. Prepare the model for k-bit training")
print("3. Configure LoRA parameters (experiment with different values)")
print("4. Apply LoRA to create the trainable model")
print("5. Check the percentage of trainable parameters (should be ~1-5%)")


## Step 6: Configure training parameters

Set up the training configuration for your QLoRA fine-tuning.


In [ ]:
# TODO: Configure training arguments
# Adjust these parameters based on your resources and needs:

# training_arguments = TrainingArguments(
#     output_dir="./results",
#     per_device_train_batch_size=1,    # Adjust based on GPU memory
#     per_device_eval_batch_size=1,
#     gradient_accumulation_steps=4,    # Effective batch size = batch_size * gradient_accumulation_steps
#     num_train_epochs=3,               # Start with 1-3 epochs
#     learning_rate=2e-4,               # LoRA typically uses higher learning rates
#     fp16=True,                        # Use mixed precision training
#     logging_steps=10,
#     evaluation_strategy="steps",
#     eval_steps=50,
#     save_strategy="epoch",
#     report_to=None,                   # Disable wandb for now
# )

print("Training configuration instructions:")
print("1. Start with small batch sizes (1-2) to avoid memory issues")
print("2. Use gradient accumulation to simulate larger batch sizes")
print("3. Begin with 1-3 epochs to avoid overfitting")
print("4. LoRA can use higher learning rates (1e-4 to 5e-4)")
print("5. Enable fp16 for memory efficiency")


## Step 7: Train the model

Execute the QLoRA fine-tuning process.


In [ ]:
# TODO: Create and run the trainer
# trainer = SFTTrainer(
#     model=model,
#     train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     peft_config=lora_config,
#     dataset_text_field="text",
#     tokenizer=tokenizer,
#     args=training_arguments,
#     max_seq_length=512,
# )

# TODO: Start training
# trainer.train()

# TODO: Save the fine-tuned model
# trainer.save_model("./fine-tuned-model")

print("Training execution instructions:")
print("1. Create the SFTTrainer with your model, dataset, and configuration")
print("2. Start training with trainer.train() - this will take time!")
print("3. Monitor the training loss - it should decrease over time")
print("4. Save the trained model and tokenizer")
print("5. Training time varies: 30 minutes to several hours depending on data size")


## Step 8: Test and compare models

Evaluate your fine-tuned model against the base model.


In [ ]:
# TODO: Load your fine-tuned model for testing
# from peft import PeftModel
# fine_tuned_model = PeftModel.from_pretrained(base_model, "./fine-tuned-model")

# TODO: Create test prompts relevant to your educational domain
test_prompts = [
    # Add 3-5 test prompts here that are relevant to your domain
    # Example: "Explain the concept of derivatives in calculus"
]

# TODO: Test both models and compare outputs
# for prompt in test_prompts:
#     print(f"\nTest Prompt: {prompt}")
#
#     # Base model response
#     base_response = # Generate response with base model
#     print(f"Base Model: {base_response}")
#
#     # Fine-tuned model response
#     ft_response = # Generate response with fine-tuned model
#     print(f"Fine-tuned Model: {ft_response}")
#     print("-" * 80)

print("Model evaluation instructions:")
print("1. Create test prompts specific to your educational domain")
print("2. Generate responses from both base and fine-tuned models")
print("3. Compare the quality, accuracy, and domain-specific knowledge")
print("4. Document which responses are more educational and helpful")
print("5. Consider factors like accuracy, pedagogical approach, and clarity")


## Step 9: Project summary and analysis

Document your findings and analyze the effectiveness of your QLoRA implementation.


### A. Educational domain and specialization goal

**Your chosen domain:** ___________________________

**Why this domain benefits from specialization:** ___________________________

**Target capabilities developed:** ___________________________

### B. Technical implementation summary

**Base model used:** ___________________________

**Dataset source and size:** ___________________________

**LoRA configuration (r, alpha, dropout):** ___________________________

**Training time and resources:** ___________________________

### C. Model performance analysis

**Training loss progression:** ___________________________

**Comparison with base model:** ___________________________

**Specific improvements observed:** ___________________________

### D. Educational value assessment

**Quality of domain-specific responses:** ___________________________

**Pedagogical effectiveness:** ___________________________

**Potential classroom applications:** ___________________________

### E. Technical challenges and solutions

**Main challenges encountered:** ___________________________

**Solutions implemented:** ___________________________

**Lessons learned:** ___________________________

### F. Future deployment considerations

**Model size and inference requirements:** ___________________________

**Integration possibilities:** ___________________________

**Scaling considerations:** ___________________________


## Submission checklist

Before submitting, ensure you have:

- [ ] Selected and justified an appropriate educational domain for specialization
- [ ] Loaded and preprocessed a suitable dataset (500-1000 examples minimum)
- [ ] Successfully implemented QLoRA fine-tuning with appropriate parameters
- [ ] Documented the training process including loss progression and parameters
- [ ] Tested and compared your fine-tuned model against the base model
- [ ] Analyzed the educational value and effectiveness of your specialized model
- [ ] Completed the project summary with technical and pedagogical insights

